# GNN Bipartite Tertile Classification — Parquet edition

Same model and methodology as `gnn_bipartite_tertile.ipynb`, but reads parquet
files instead of querying PostgreSQL. Designed to run on the BGU cluster.

- Class 0 = bottom third of next-quarter outcome
- Class 1 = middle third
- Class 2 = top third

Stocks are bucketed by next-quarter `log_return` terciles; investors by
next-quarter profitability terciles. Random-chance accuracy is 1/3 (0.333).

## Paths

All outputs (models, results CSV, pickles) are written under `OUT_DIR`, which
defaults to `~/13Fgnn/`. Parquet inputs are read from `DATA_DIR`, which defaults
to `~/13Fgnn/data/`. Override either with the env vars `FGNN_OUT_DIR` /
`FGNN_DATA_DIR`.

## Setup

### torch / cuda check

In [ ]:
# pip install torch --index-url https://download.pytorch.org/whl/cu124
# pip install torch_geometric

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("cuda build:", torch.version.cuda)

### Imports & paths

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GATConv

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# OUT_DIR holds everything we produce on the cluster: models/, results CSV, etc.
# DATA_DIR holds the parquet inputs uploaded from the local export notebook.
OUT_DIR = Path(os.environ.get("FGNN_OUT_DIR", str(Path.home() / "13Fgnn"))).expanduser().resolve()
DATA_DIR = Path(os.environ.get("FGNN_DATA_DIR", str(OUT_DIR / "data"))).expanduser().resolve()

MODELS_DIR = OUT_DIR / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"DATA_DIR not found: {DATA_DIR}. Upload parquets there or set FGNN_DATA_DIR."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
torch.manual_seed(0)
np.random.seed(0)

print("OUT_DIR    :", OUT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("MODELS_DIR :", MODELS_DIR)
print("device     :", DEVICE)

### Load parquets

We load each table once into memory; the loader functions below filter by
`(year, quarter)` in pandas instead of issuing SQL `WHERE`.

In [ ]:
TGT_CHANGED_HOLDINGS = "changed_holdings"
TGT_STOCKS_RETURN = "stocks_return"
TGT_CIK_AUM = "cik_aum"
TGT_NORMALIZED_HOLDINGS = "normalized_holdings"
CUSIP_FIN_TABLE = "cusip_financial_data"


def _read_parquet_or_empty(name: str) -> pd.DataFrame:
    p = DATA_DIR / f"{name}.parquet"
    if not p.exists():
        print(f"  [warn] missing parquet: {p.name} (returning empty df)")
        return pd.DataFrame()
    df = pd.read_parquet(p)
    print(f"  loaded {name:30s} {len(df):>10,} rows  {len(df.columns):>3} cols")
    return df


CHANGED_HOLDINGS = _read_parquet_or_empty(TGT_CHANGED_HOLDINGS)
STOCKS_RETURN = _read_parquet_or_empty(TGT_STOCKS_RETURN)
CIK_AUM = _read_parquet_or_empty(TGT_CIK_AUM)
NORM_HOLDINGS = _read_parquet_or_empty(TGT_NORMALIZED_HOLDINGS)
CUSIP_FIN = _read_parquet_or_empty(CUSIP_FIN_TABLE)

### Configuration of the GNN, and global vars

In [ ]:
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]

# 3 options for edges:
# EDGES_COLUMN_NAME = "change_in_adjusted_weight"
# EDGES_COLUMN_NAME = "change_in_shares"
EDGES_COLUMN_NAME = "change_in_weight"

In [ ]:
YEAR, QUARTER = 2019, 3

NUM_CLASSES = 3
HIDDEN_DIM = 64
NUM_LAYERS = 2
EPOCHS = 150
LR = 8e-4
DROPOUT = 0.5
MASK_FRAC = 0.20
GAT_HEADS = 4

## Data loaders (pandas / parquet)

Drop-in replacements for the original SQL loaders. Same return shapes, same
column names — everything downstream stays identical.

In [ ]:
def next_year_quarter(year: int, quarter: int):
    return (year + 1, 1) if quarter == 4 else (year, quarter + 1)


def load_edges(year, quarter, edges_col_name):
    df = CHANGED_HOLDINGS
    if df.empty or edges_col_name not in df.columns:
        return pd.DataFrame(columns=["cik", "cusip", "w"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df[edges_col_name].notna()
    out = df.loc[mask, ["cik", "cusip", edges_col_name]].rename(columns={edges_col_name: "w"})
    return out.reset_index(drop=True)


def load_returns(year, quarter):
    df = STOCKS_RETURN
    if df.empty:
        return pd.DataFrame(columns=["cusip", "log_return"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df["log_return"].notna()
    return df.loc[mask, ["cusip", "log_return"]].reset_index(drop=True)


def load_aum(year, quarter):
    df = CIK_AUM
    if df.empty:
        return pd.DataFrame(columns=["cik", "aum"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & (df["total"] > 0)
    out = df.loc[mask, ["cik", "total"]].rename(columns={"total": "aum"})
    return out.reset_index(drop=True)


def load_stock_features(year, quarter) -> pd.DataFrame:
    fin = CUSIP_FIN
    if fin.empty:
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    else:
        fin = fin.loc[(fin["year"] == year) & (fin["quarter"] == quarter)].copy()
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]


def investor_profitability(year, quarter) -> pd.Series:
    ny, nq = next_year_quarter(year, quarter)
    h = NORM_HOLDINGS
    if h.empty:
        return pd.Series(dtype=float, name="profitability")
    h = h.loc[(h["year"] == year) & (h["quarter"] == quarter), ["cik", "cusip", "weight"]]
    r = load_returns(ny, nq)
    m = h.merge(r, on="cusip", how="inner")
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")

## Tertile labeller

`qcut` into 3 equal-frequency buckets. NaN values → −1 (masked out of loss).
Ties get spread by `duplicates='drop'` fallback to numeric thresholds.

In [ ]:
def tertile_labels(values: pd.Series) -> pd.Series:
    """Return integer labels in {0, 1, 2} via tertile cut. NaN -> -1."""
    s = values.astype(float)
    out = pd.Series(-1, index=s.index, dtype=np.int64)
    valid = s.dropna()
    if valid.empty:
        return out
    try:
        cats = pd.qcut(valid, q=3, labels=[0, 1, 2])
    except ValueError:
        q1, q2 = np.quantile(valid, [1 / 3, 2 / 3])
        cats = pd.cut(valid, bins=[-np.inf, q1, q2, np.inf], labels=[0, 1, 2], include_lowest=True)
    out.loc[valid.index] = cats.astype(np.int64)
    return out

## Graph builder

Build a graph for `(year, quarter)`. Edges are driven by `EDGES_COLUMN_NAME`.

In [ ]:
def zscore(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        v = out[c].astype(float)
        v = v.replace([np.inf, -np.inf], np.nan).fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out

In [ ]:
# ---- 1) Investor (CIK) feature table ---------------------------------------
def _build_cik_features(edges, year, quarter):
    """One row per CIK with [log_aum, n_holdings, profitability], z-scored.
    Uses past-quarter profitability as a feature (no leakage)."""
    aum = load_aum(year, quarter)

    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])

    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()

    cik_df = (aum.merge(cik_nh, on="cik", how="outer")
                  .merge(past_prof, on="cik", how="left"))

    cik_df["aum"] = cik_df["aum"].fillna(
        cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)

    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    return zscore(cik_df, CIK_FEATS), CIK_FEATS


# ---- 2) Stock (CUSIP) feature table ----------------------------------------
def _build_stock_features(year, quarter):
    stock_df = load_stock_features(year, quarter)
    return zscore(stock_df, STOCK_FEATURE_COLS)


# ---- 3) Forward-looking outcomes used as labels ----------------------------
def _load_label_sources(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    r_next = load_returns(ny, nq).set_index("cusip")["log_return"]
    prof_next = investor_profitability(year, quarter)
    return r_next, prof_next


# ---- 4) Build node feature matrix X with CIKs first, CUSIPs second ---------
def _assemble_node_matrix(edges, cik_df, stock_df, CIK_FEATS):
    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())

    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)

    F_cik = cik_df[CIK_FEATS].to_numpy()
    F_stk = stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]

    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk

    cik_pos = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}
    return X, cik_ids, cusip_ids, cik_pos, cusip_pos


# ---- 5) Build undirected edge_index + edge_weight --------------------------
def _assemble_edges(edges, cik_pos, cusip_pos):
    src = edges["cik"].map(cik_pos).to_numpy()
    dst = edges["cusip"].map(cusip_pos).to_numpy()
    edge_index = np.stack(
        [np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    edge_weight = np.concatenate(
        [edges["w"].to_numpy(), edges["w"].to_numpy()]).astype(np.float32)
    return edge_index, edge_weight


# ---- 6) Assign tertile labels (3 classes, -1 = unknown) --------------------
def _assign_labels(num_nodes, cusip_pos, cik_pos, r_next, prof_next):
    y = np.full(num_nodes, -1, dtype=np.int64)
    if not r_next.empty:
        stk_lbl = tertile_labels(r_next)
        for cusip, idx in cusip_pos.items():
            v = stk_lbl.get(cusip, -1)
            if v >= 0:
                y[idx] = int(v)
    if not prof_next.empty:
        inv_lbl = tertile_labels(prof_next)
        for cik, idx in cik_pos.items():
            v = inv_lbl.get(cik, -1)
            if v >= 0:
                y[idx] = int(v)
    return y


# ---- 7) Orchestrator -------------------------------------------------------
def build_graph(year: int, quarter: int, edges_col_name):
    edges = load_edges(year, quarter, edges_col_name)
    if edges.empty:
        raise RuntimeError(f"no \u0394-edges for {year} Q{quarter}")

    cik_df, CIK_FEATS = _build_cik_features(edges, year, quarter)
    stock_df = _build_stock_features(year, quarter)
    r_next, prof_next = _load_label_sources(year, quarter)

    X, cik_ids, cusip_ids, cik_pos, cusip_pos = _assemble_node_matrix(
        edges, cik_df, stock_df, CIK_FEATS)

    edge_index, edge_weight = _assemble_edges(edges, cik_pos, cusip_pos)
    y = _assign_labels(X.shape[0], cusip_pos, cik_pos, r_next, prof_next)

    data = Data(
        x=torch.tensor(X),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        edge_weight=torch.tensor(edge_weight),
        y=torch.tensor(y),
    )
    data.is_cik = torch.zeros(X.shape[0], dtype=torch.bool)
    data.is_cik[:len(cik_ids)] = True
    data.has_label = data.y >= 0

    meta = {
        "cik_ids": cik_ids, "cusip_ids": cusip_ids,
        "n_cik": len(cik_ids), "n_cusip": len(cusip_ids),
    }
    return data, meta

In [ ]:
# quick exploration of the inputs at one quarter
edges = load_edges(2016, 3, EDGES_COLUMN_NAME)
cik_df, _ = _build_cik_features(edges, 2016, 3)
stock_df = _build_stock_features(2016, 3)
r_next, prof_next = _load_label_sources(2016, 3)

In [ ]:
cik_df.head()

In [ ]:
stock_df.head()

In [ ]:
data, meta = build_graph(YEAR, QUARTER, EDGES_COLUMN_NAME)
print(f"nodes: {data.num_nodes:,}  (CIKs {meta['n_cik']:,}, CUSIPs {meta['n_cusip']:,})")
print(f"edges: {data.num_edges:,}  (undirected, doubled)")
print(f"feat dim: {data.num_node_features}")
labeled = data.has_label
print(f"labeled nodes: {int(labeled.sum()):,}")
print("class distribution (labeled):",
      {int(c): int((data.y[labeled] == c).sum()) for c in [0, 1, 2]})

## Models

In [ ]:
class WeightedSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES, num_layers=2, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)

In [ ]:
class WeightedGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES, num_layers=2, heads=4, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout))
        self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)

## Training loop & eval

Fixed `EPOCHS=150`. After every epoch we keep the snapshot with the best
validation accuracy and restore those weights at the end.

In [ ]:
def train_one(model, data, train_mask, val_mask, epochs=EPOCHS, lr=LR, verbose=False):
    model = model.to(DEVICE)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask = val_mask.to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    best_val_acc = 0.0
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        logits = model(data.x, data.edge_index, data.edge_weight)
        loss = F.cross_entropy(logits[train_mask], data.y[train_mask])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index, data.edge_weight)
            pred = logits.argmax(dim=-1)
            val_acc = (
                (pred[val_mask] == data.y[val_mask]).float().mean().item()
                if val_mask.any() else 0.0
            )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        if verbose and ep % 25 == 0:
            print(f"  ep {ep:3d}  loss={loss.item():.4f}  val_acc={val_acc:.3f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

In [ ]:
def eval_subsets(model, data, mask):
    model.eval()
    with torch.no_grad():
        logits = model(
            data.x.to(DEVICE),
            data.edge_index.to(DEVICE),
            data.edge_weight.to(DEVICE),
        )
        pred = logits.argmax(dim=-1).cpu()

    y = data.y.cpu()
    out = {}
    for label, sel in [
        ("all", mask),
        ("stocks", mask & (~data.is_cik.cpu())),
        ("investors", mask & data.is_cik.cpu()),
    ]:
        if sel.sum() == 0:
            out[label] = {"n": 0}
            continue
        yt = y[sel].numpy()
        yp = pred[sel].numpy()
        out[label] = {
            "n": int(sel.sum()),
            "accuracy": accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro", labels=[0, 1, 2], zero_division=0),
        }
    return out

## Node-mask methodology

Random 20% of labeled nodes held out on the same graph. Random-chance acc = 0.333.

In [ ]:
def make_mask_split(data, frac=MASK_FRAC, seed=0):
    rng = np.random.default_rng(seed)
    labeled = data.has_label.cpu().nonzero(as_tuple=True)[0].numpy()
    perm = rng.permutation(labeled)
    cut = int(len(perm) * (1 - frac))
    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    train_mask[perm[:cut]] = True
    test_mask[perm[cut:]] = True
    return train_mask, test_mask

In [ ]:
train_mask, test_mask = make_mask_split(data)
print(f"train labeled: {int(train_mask.sum()):,}   test labeled: {int(test_mask.sum()):,}")

sage = WeightedSAGE(data.num_node_features, HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT)
sage = train_one(sage, data, train_mask, test_mask, verbose=True)
res_sage_mask = eval_subsets(sage, data, test_mask)

gat = WeightedGAT(data.num_node_features, HIDDEN_DIM, num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT)
gat = train_one(gat, data, train_mask, test_mask, verbose=True)
res_gat_mask = eval_subsets(gat, data, test_mask)

results_df = pd.DataFrame({
    "GraphSAGE": {f"{k}.{m}": v for k, d in res_sage_mask.items() for m, v in d.items()},
    "GAT": {f"{k}.{m}": v for k, d in res_gat_mask.items() for m, v in d.items()},
})
results_df

## Two-periods methodology

Train on Δ at (year, quarter−1), test on Δ at (year, quarter).

In [ ]:
def two_periods_eval(year, quarter, model_cls, edges_col_name, **kwargs):
    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)
    train_data, _ = build_graph(py, pq, edges_col_name)
    test_data, _ = build_graph(year, quarter, edges_col_name)

    Fdim = max(train_data.num_node_features, test_data.num_node_features)

    def pad(d):
        if d.num_node_features < Fdim:
            pad_cols = torch.zeros(d.num_nodes, Fdim - d.num_node_features)
            d.x = torch.cat([d.x, pad_cols], dim=1)
        return d

    train_data = pad(train_data)
    test_data = pad(test_data)

    model = model_cls(Fdim, HIDDEN_DIM, **kwargs)
    model = train_one(model, train_data, train_data.has_label, train_data.has_label, verbose=False)
    return eval_subsets(model, test_data, test_data.has_label)

In [ ]:
res_sage_2p = two_periods_eval(YEAR, QUARTER, WeightedSAGE, EDGES_COLUMN_NAME, num_layers=NUM_LAYERS, dropout=DROPOUT)
res_gat_2p = two_periods_eval(YEAR, QUARTER, WeightedGAT, EDGES_COLUMN_NAME, num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT)

pd.DataFrame({
    "GraphSAGE": {f"{k}.{m}": v for k, d in res_sage_2p.items() for m, v in d.items()},
    "GAT": {f"{k}.{m}": v for k, d in res_gat_2p.items() for m, v in d.items()},
})

## Summary

In [ ]:
def flat(res, tag):
    return {
        f"{tag}.all.acc": res["all"].get("accuracy"),
        f"{tag}.all.f1":  res["all"].get("macro_f1"),
        f"{tag}.stocks.acc": res["stocks"].get("accuracy"),
        f"{tag}.investors.acc": res["investors"].get("accuracy"),
    }

summary = pd.DataFrame([
    {"method": "GraphSAGE (mask)",        **flat(res_sage_mask, "test")},
    {"method": "GAT (mask)",              **flat(res_gat_mask,  "test")},
    {"method": "GraphSAGE (two-periods)", **flat(res_sage_2p,   "test")},
    {"method": "GAT (two-periods)",       **flat(res_gat_2p,    "test")},
])
print("Random-chance accuracy = 0.333")
summary.to_csv(OUT_DIR / "summary.csv", index=False)
summary

## Per-quarter sweep — two-periods evaluation

Iterates every (year, quarter) where Δ-graphs at (t-1) and (t+1) both exist,
runs the two-periods protocol for SAGE and GAT, and stores per-quarter results.
Resumable: re-running skips quarters already in the CSV.

Outputs (under `OUT_DIR`):
- `models/two_periods_quarter_results.csv` — full sweep table
- `models/best_<tag>_<year>Q<q>.pkl` — pickled best model + config

In [ ]:
import time
import pickle
import gc

RESULTS_CSV = MODELS_DIR / "two_periods_quarter_results.csv"


def list_available_quarters():
    """Quarters where both (t-1) and (t+1) exist in changed_holdings."""
    df = CHANGED_HOLDINGS
    sub = df.loc[df[EDGES_COLUMN_NAME].notna(), ["year", "quarter"]].drop_duplicates()
    yq = sorted({(int(y), int(q)) for y, q in sub.itertuples(index=False)})
    avail = set(yq)
    out = []
    for y, q in yq:
        py, pq = (y - 1, 4) if q == 1 else (y, q - 1)
        ny, nq = next_year_quarter(y, q)
        if (py, pq) in avail and (ny, nq) in avail:
            out.append((y, q))
    return out


def two_periods_with_model(year, quarter, model_cls, edges_col_name, **kwargs):
    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)
    train_data, _ = build_graph(py, pq, edges_col_name)
    test_data, _ = build_graph(year, quarter, edges_col_name)

    Fdim = max(train_data.num_node_features, test_data.num_node_features)

    def pad(d):
        if d.num_node_features < Fdim:
            pad_cols = torch.zeros(d.num_nodes, Fdim - d.num_node_features)
            d.x = torch.cat([d.x, pad_cols], dim=1)
        return d

    train_data = pad(train_data)
    test_data = pad(test_data)

    model = model_cls(Fdim, HIDDEN_DIM, **kwargs)
    model = train_one(model, train_data, train_data.has_label, train_data.has_label, verbose=False)
    results = eval_subsets(model, test_data, test_data.has_label)

    train_data = train_data.cpu()
    test_data = test_data.cpu()
    model = model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, results, Fdim


def load_done_set(csv_path):
    if not csv_path.exists():
        return set(), pd.DataFrame()
    df = pd.read_csv(csv_path)
    done = set(zip(df["year"].astype(int), df["quarter"].astype(int)))
    return done, df


def append_row_to_csv(row, csv_path):
    pd.DataFrame([row]).to_csv(
        csv_path, mode="a", header=not csv_path.exists(), index=False)


def aggressive_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def run_sweep(quarters, csv_path, best):
    done, _ = load_done_set(csv_path)
    remaining = [(y, q) for y, q in quarters if (y, q) not in done]
    print(f"resume: {len(done)} done, {len(remaining)} remaining")

    t_start = time.perf_counter()
    for i, (y, q) in enumerate(remaining, 1):
        row = {"year": y, "quarter": q}

        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.perf_counter()
            m, r, fd = two_periods_with_model(
                y, q, WeightedSAGE, EDGES_COLUMN_NAME,
                num_layers=NUM_LAYERS, dropout=DROPOUT,
            )
            row["sage_train_s"] = time.perf_counter() - t0
            row["sage_peak_gb"] = (torch.cuda.max_memory_allocated() / 1e9
                                    if torch.cuda.is_available() else 0.0)
            row["sage_acc"] = r["all"].get("accuracy")
            row["sage_f1"] = r["all"].get("macro_f1")
            row["sage_stocks_acc"] = r["stocks"].get("accuracy")
            row["sage_inv_acc"] = r["investors"].get("accuracy")
            if row["sage_acc"] is not None and row["sage_acc"] > best["acc"]:
                best.update({"acc": row["sage_acc"], "model": m, "tag": "sage",
                              "Fdim": fd, "year": y, "quarter": q})
        except Exception as e:
            print(f"  ! SAGE {y} Q{q} failed: {e.__class__.__name__}: {e}")
        finally:
            aggressive_cleanup()

        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.perf_counter()
            m, r, fd = two_periods_with_model(
                y, q, WeightedGAT, EDGES_COLUMN_NAME,
                num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT,
            )
            row["gat_train_s"] = time.perf_counter() - t0
            row["gat_peak_gb"] = (torch.cuda.max_memory_allocated() / 1e9
                                   if torch.cuda.is_available() else 0.0)
            row["gat_acc"] = r["all"].get("accuracy")
            row["gat_f1"] = r["all"].get("macro_f1")
            row["gat_stocks_acc"] = r["stocks"].get("accuracy")
            row["gat_inv_acc"] = r["investors"].get("accuracy")
            if row["gat_acc"] is not None and row["gat_acc"] > best["acc"]:
                best.update({"acc": row["gat_acc"], "model": m, "tag": "gat",
                              "Fdim": fd, "year": y, "quarter": q})
        except Exception as e:
            print(f"  ! GAT {y} Q{q} failed: {e.__class__.__name__}: {e}")
        finally:
            aggressive_cleanup()

        append_row_to_csv(row, csv_path)

        elapsed = time.perf_counter() - t_start
        eta = elapsed / i * (len(remaining) - i)
        free_gb = (torch.cuda.mem_get_info()[0] / 1e9
                   if torch.cuda.is_available() else 0.0)
        print(f"  [{len(done) + i:2d}/{len(quarters)}] {y} Q{q}  "
              f"SAGE={row.get('sage_acc', float('nan')):.3f} "
              f"(peak {row.get('sage_peak_gb', 0):.2f}GB)  "
              f"GAT={row.get('gat_acc', float('nan')):.3f} "
              f"(peak {row.get('gat_peak_gb', 0):.2f}GB)  "
              f"free {free_gb:.2f}GB  ETA {eta/60:.1f}m")


quarters = list_available_quarters()
print(f"running two-periods on {len(quarters)} quarters: {quarters[0]} … {quarters[-1]}")

MAX_ATTEMPTS = 3
best = {"acc": -1.0, "model": None, "tag": None, "Fdim": None, "year": None, "quarter": None}
overall_start = time.perf_counter()
last_error = None

for attempt in range(1, MAX_ATTEMPTS + 1):
    print(f"\n=== attempt {attempt}/{MAX_ATTEMPTS} ===")
    try:
        run_sweep(quarters, RESULTS_CSV, best)
        last_error = None
        done_now, _ = load_done_set(RESULTS_CSV)
        if len(done_now) >= len(quarters):
            print("all quarters complete.")
            break
    except KeyboardInterrupt:
        raise
    except Exception as e:
        last_error = e
        print(f"!! attempt {attempt} crashed: {e.__class__.__name__}: {e}")
        aggressive_cleanup()
        time.sleep(5)
        continue

if last_error is not None:
    raise RuntimeError(
        f"sweep failed after {MAX_ATTEMPTS} attempts; last error: "
        f"{last_error.__class__.__name__}: {last_error}"
    )

quarter_results_df = pd.read_csv(RESULTS_CSV)
print(f"\nfinished in {(time.perf_counter() - overall_start)/60:.1f} min")
print(f"best model: {best['tag']} on {best['year']} Q{best['quarter']}  (acc={best['acc']:.3f})")

if best["model"] is not None:
    best_path = MODELS_DIR / f"best_{best['tag']}_{best['year']}Q{best['quarter']}.pkl"
    with open(best_path, "wb") as f:
        pickle.dump({
            "tag": best["tag"],
            "year": best["year"],
            "quarter": best["quarter"],
            "Fdim": best["Fdim"],
            "test_acc": best["acc"],
            "state_dict": {k: v.cpu() for k, v in best["model"].state_dict().items()},
            "config": {
                "HIDDEN_DIM": HIDDEN_DIM,
                "NUM_LAYERS": NUM_LAYERS,
                "GAT_HEADS": GAT_HEADS,
                "DROPOUT": DROPOUT,
                "NUM_CLASSES": NUM_CLASSES,
                "EDGES_COLUMN_NAME": EDGES_COLUMN_NAME,
            },
        }, f)
    print(f"saved → {best_path}")
else:
    print("no best model captured this run (only resumed previously-saved rows).")

print(f"results CSV → {RESULTS_CSV}")
quarter_results_df

In [ ]:
# Aggregate summary across all quarters
acc_cols  = ["sage_acc", "gat_acc"]
f1_cols   = ["sage_f1",  "gat_f1"]
time_cols = ["sage_train_s", "gat_train_s"]

print(f"quarters run: {len(quarter_results_df)}   |   random-chance acc = {1/NUM_CLASSES:.3f}\n")

print("accuracy  mean +/- std:")
print(quarter_results_df[acc_cols].agg(["mean", "std"]).T.round(3))

print("\nmacro-F1  mean +/- std:")
print(quarter_results_df[f1_cols].agg(["mean", "std"]).T.round(3))

print("\ntrain time (s) mean:")
print(quarter_results_df[time_cols].mean().round(2))

print("\ntop-5 quarters by best (max(SAGE, GAT)) accuracy:")
top5 = (quarter_results_df.assign(best=quarter_results_df[acc_cols].max(axis=1))
        .sort_values("best", ascending=False)
        .head(5)[["year", "quarter", "sage_acc", "gat_acc", "best"]])
top5.to_csv(OUT_DIR / "top5_quarters.csv", index=False)
top5